# Multivariate fine-mapping with mvSuSiE

This vignette shows how to jointly fine-map the genetic effects of one molecular trait measured across multiple contexts.

## Learning goals

After completing this vignette, you will be able to:

- distinguish cross-context mvSuSiE from single-context SuSiE and cross-gene fine-mapping;
- prepare matched genotype, phenotype and covariate inputs;
- run the current `qtl_dataset_construct+mnm` workflow; and
- interpret context-specific posterior effects together with shared variant PIPs.

## Background and method

When the same molecular trait is measured in several tissues, cell types or experimental conditions, analyzing each context separately discards information about shared genetic effects. The `mnm` workflow uses mvSuSiE to model the contexts jointly. It estimates a multivariate effect for each candidate variant and uses patterns learned across contexts to distinguish effects that are shared, context-specific or too weak to resolve.

The workflow first harmonizes genotype, phenotype and covariate data in a `QtlDataset`. It then fine-maps each requested trait across its available contexts. By default, mvSuSiE uses its canonical prior. A previously fitted TWAS-weight object can be supplied with `--prior-twas-weights` to provide a data-driven prior; this is optional and should not be confused with running a separate mr.mash analysis inside this workflow.

Use this route for **one trait across contexts**. Use ordinary `susie_twas` for one trait in one context, and use `mnm_genes` when the scientific unit is multiple genes within the same locus.

## Worked example

The example fine-maps `ENSG00000130538` jointly in two contexts from the same individuals.

| Role | Example path | What it supplies |
|---|---|---|
| Genotype | `input/finemapping/protocol_example.genotype.chr22.bed` | PLINK bed/bim/fam trio |
| Phenotype manifest | `input/finemapping/protocol_example.pheno_manifest_multicontext.tsv` | One manifest row per context and its phenotype BED file |
| Covariates | `input/finemapping/protocol_example.covariates.tsv` | Covariates for the shared samples |
| Association windows | `input/finemapping/protocol_example.association_windows.bed` | Coordinates for the requested trait |

Context rows must refer to the same trait, genome build and sample population. Sample identifiers must agree across all files.

### [Run cross-context fine-mapping](https://statfungen.github.io/xqtl-protocol/mnm_regression.html#multivariate-fine-mapping)

`qtl_dataset_construct+mnm` constructs the harmonized dataset and performs one joint mvSuSiE fit for the selected trait.

**Timing**: TBD

In [ ]:
sos run pipeline/mnm_regression.ipynb qtl_dataset_construct+mnm \
  --name protocol_example \
  --cwd output_mnm \
  --genoFile input/finemapping/protocol_example.genotype.chr22.bed \
  --phenoFile input/finemapping/protocol_example.pheno_manifest_multicontext.tsv \
  --covFile input/finemapping/protocol_example.covariates.tsv \
  --customized-association-windows input/finemapping/protocol_example.association_windows.bed \
  --region-name ENSG00000130538 \
  --transpose-covariates \
  -j 1

The command is expected to create:

- `output_mnm/qtl_dataset/protocol_example.qtl_dataset.rds` — harmonized genotype, phenotype and covariate data.
- `output_mnm/multivariate_fine_mapping/protocol_example.ENSG00000130538.multicontext_bvsr.rds` — the cross-context `QtlFineMappingResult`.

The most useful optional settings are:

| Option | Purpose |
|---|---|
| `--contexts` | Restrict the fit to named contexts; empty uses all available contexts |
| `--mvsusie-max-iter` | Maximum mvSuSiE iterations; default `200` |
| `--prior-twas-weights` | Optional preceding TWAS-weight RDS used to derive a data-driven prior |
| `--prior-weights-min` | Lower bound applied to supplied prior weights |
| `--mvsusie-method-args` | Additional mvSuSiE arguments as JSON |

### Command reference

Check the current interface before adapting the command to a new study.

In [ ]:
sos run pipeline/mnm_regression.ipynb -h

## Results and interpretation

### Cross-context fine-mapping result

The output is a `QtlFineMappingResult`. The result table identifies the fitted contexts and method; each `entry` contains the mvSuSiE fit and an attached `topLoci` table.

| Component | Interpretation |
|---|---|
| Result metadata | Trait, contexts and fitted method |
| `topLoci` | Variant PIPs, posterior effect summaries and credible-set membership |
| mvSuSiE fit | Context-specific posterior effects for each single-effect component |
| Credible-set purity | Whether variants in a credible set are sufficiently correlated to support localization |

In [ ]:
suppressPackageStartupMessages(library(pecotmr))

result_file <- "output_mnm/multivariate_fine_mapping/protocol_example.ENSG00000130538.multicontext_bvsr.rds"
result <- readRDS(result_file)

cat("Class:", class(result), "\n")
print(result[, setdiff(names(result), "entry"), drop = FALSE])

entry <- result$entry[[1]]
top_loci <- getTopLoci(entry)
top_loci <- top_loci[order(top_loci$pip, decreasing = TRUE), ]
keep <- intersect(c("variant_id", "pip", "cs_95", "cs_95_purity", "method"),
                  names(top_loci))
head(top_loci[, keep, drop = FALSE], 8)

Rank variants by PIP, then evaluate credible-set size and purity. The multivariate fit can strengthen evidence when effects are consistent across contexts, but a high joint PIP does not mean that the effect has the same magnitude in every context. Inspect the context-specific posterior effects before labeling a variant as shared or context-specific. Broad, low-purity credible sets indicate insufficient localization rather than a single confirmed causal variant.

## Limitations and common pitfalls

- All context-specific phenotypes must represent the same trait and use compatible genomic coordinates.
- mvSuSiE does not correct sample mismatches; genotype, phenotype and covariate identifiers must be harmonized first.
- A missing phenotype in some contexts changes the information available to the joint model and should be documented.
- A data-driven prior is optional. Do not supply an unrelated TWAS-weight object merely to enable the option.
- The toy dataset demonstrates the interface but is too small for biological conclusions.
- Joint evidence does not replace examination of context-specific effect sizes and uncertainty.

## Next steps

For well-localized signals, compare posterior effects across contexts and annotate the highest-PIP variants. Use the cross-gene `mnm_genes` workflow only when the next question concerns several genes in the same locus, and use downstream post-processing to summarize credible sets across analyses.